In [0]:
# ============================================================
# Ajay | Notebook 3: Bronze → Silver Delta Table
# Explode, clean and type-cast all 10,676 aircraft records
# ============================================================

dbutils.widgets.text("catalog", "dev_team")
dbutils.widgets.text("schema",  "testing")
CATALOG       = dbutils.widgets.get("catalog")
SCHEMA        = dbutils.widgets.get("schema")
BRONZE_TABLE  = f"`{CATALOG}`.`{SCHEMA}`.Ajay_bronze_flight_raw"
SILVER_TABLE  = f"`{CATALOG}`.`{SCHEMA}`.Ajay_silver_flight_positions"

print(f"✅ Config loaded")
print(f"   Bronze : {BRONZE_TABLE}")
print(f"   Silver : {SILVER_TABLE}")

✅ Config loaded
   Bronze : `dev_team`.`testing`.Ajay_bronze_flight_raw
   Silver : `dev_team`.`testing`.Ajay_silver_flight_positions


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, DoubleType, LongType, BooleanType, IntegerType
)

# Read Bronze table
print("📖 Reading Bronze table...")
bronze_df = spark.table(BRONZE_TABLE)

# Explode the states array — 1 row becomes 10,676 rows!
print("💥 Exploding states array into individual aircraft rows...")
exploded_df = bronze_df.select(
    F.col("time").alias("api_response_time"),
    F.col("insert_dttm").alias("bronze_insert_dttm"),
    F.explode(F.col("states")).alias("state")
)

print(f"✅ Explosion complete!")
print(f"   📊 Rows before : {bronze_df.count()}")
print(f"   📊 Rows after  : {exploded_df.count()}")
print(f"\n🔍 Sample row:")
exploded_df.show(2, truncate=80)

📖 Reading Bronze table...
💥 Exploding states array into individual aircraft rows...
✅ Explosion complete!
   📊 Rows before : 1
   📊 Rows after  : 10676

🔍 Sample row:
+-----------------+--------------------------+--------------------------------------------------------------------------------+
|api_response_time|        bronze_insert_dttm|                                                                           state|
+-----------------+--------------------------+--------------------------------------------------------------------------------+
|       1778003050|2026-05-05 18:00:57.763305|[39de4f, TVF234P , France, 1778003050, 1778003050, -4.1238, 37.6724, 8633.46,...|
|       1778003050|2026-05-05 18:00:57.763305|[39de4e, TVF539X , France, 1778003049, 1778003050, 4.416, 48.6468, 8724.9, fa...|
+-----------------+--------------------------+--------------------------------------------------------------------------------+
only showing top 2 rows


In [0]:
# Map each positional index to a named, typed column
# OpenSky states array positions:
# [0]=icao24, [1]=callsign, [2]=origin_country, [3]=time_position,
# [4]=last_contact, [5]=longitude, [6]=latitude, [7]=baro_altitude,
# [8]=on_ground, [9]=velocity, [10]=true_track, [11]=vertical_rate,
# [12]=sensors(skip), [13]=geo_altitude, [14]=squawk, [15]=spi,
# [16]=position_source

silver_df = exploded_df.select(
    # String columns (trimmed)
    F.trim(F.col("state")[0].cast(StringType())).alias("icao24"),
    F.trim(F.col("state")[1].cast(StringType())).alias("callsign"),
    F.trim(F.col("state")[2].cast(StringType())).alias("origin_country"),

    # Unix timestamps → readable timestamp
    F.to_timestamp(F.col("state")[3].cast(LongType())).alias("time_position"),
    F.to_timestamp(F.col("state")[4].cast(LongType())).alias("last_contact"),

    # Numeric columns
    F.col("state")[5].cast(DoubleType()).alias("longitude"),
    F.col("state")[6].cast(DoubleType()).alias("latitude"),
    F.col("state")[7].cast(DoubleType()).alias("baro_altitude"),
    F.col("state")[8].cast(BooleanType()).alias("on_ground"),
    F.col("state")[9].cast(DoubleType()).alias("velocity"),
    F.col("state")[10].cast(DoubleType()).alias("true_track"),
    F.col("state")[11].cast(DoubleType()).alias("vertical_rate"),
    F.col("state")[13].cast(DoubleType()).alias("geo_altitude"),

    # Misc columns
    F.trim(F.col("state")[14].cast(StringType())).alias("squawk"),
    F.col("state")[15].cast(BooleanType()).alias("spi"),
    F.col("state")[16].cast(IntegerType()).alias("position_source"),

    # API metadata
    F.to_timestamp(F.col("api_response_time").cast(LongType())).alias("api_response_time"),
    F.col("bronze_insert_dttm")
)

print(f"✅ Columns mapped successfully!")
print(f"   📋 Total columns : {len(silver_df.columns)}")
print(f"\n🔍 Sample aircraft data:")
silver_df.select(
    "icao24","callsign","origin_country",
    "baro_altitude","on_ground","velocity"
).show(5, truncate=20)

✅ Columns mapped successfully!
   📋 Total columns : 18

🔍 Sample aircraft data:
+------+--------+--------------+-------------+---------+--------+
|icao24|callsign|origin_country|baro_altitude|on_ground|velocity|
+------+--------+--------------+-------------+---------+--------+
|39de4f| TVF234P|        France|      8633.46|    false|  232.15|
|39de4e| TVF539X|        France|       8724.9|    false|   207.4|
|ab1644| UAL2041| United States|     11269.98|    false|  283.67|
|a47559|  N387AP| United States|      1036.32|    false|   62.81|
|39de4a| TVF87TP|        France|     11590.02|    false|  255.59|
+------+--------+--------------+-------------+---------+--------+
only showing top 5 rows


In [0]:
# Data quality check — remove records where icao24 is null
before = silver_df.count()

silver_clean_df = silver_df.filter(F.col("icao24").isNotNull())

after   = silver_clean_df.count()
dropped = before - after

print(f"🧹 Data Quality Check:")
print(f"   Rows before filter : {before}")
print(f"   Rows after filter  : {after}")
print(f"   Null icao24 dropped: {dropped}")

# Add audit columns
silver_final_df = silver_clean_df \
    .withColumn("insert_dttm", F.current_timestamp()) \
    .withColumn("update_dttm", F.current_timestamp()) \
    .withColumn("inserted_by", F.current_user()) \
    .withColumn("updated_by",  F.current_user())

print(f"\n✅ Audit columns added!")
print(f"   📋 Total columns : {len(silver_final_df.columns)}")

🧹 Data Quality Check:
   Rows before filter : 10676
   Rows after filter  : 10676
   Null icao24 dropped: 0

✅ Audit columns added!
   📋 Total columns : 22


In [0]:
# Write to Silver Delta table
print(f"💾 Writing to Silver table: {SILVER_TABLE}")

silver_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

row_count = spark.table(SILVER_TABLE).count()

print(f"✅ Silver table written successfully!")
print(f"   📋 Table     : {SILVER_TABLE}")
print(f"   📊 Row count : {row_count}")

💾 Writing to Silver table: `dev_team`.`testing`.Ajay_silver_flight_positions
✅ Silver table written successfully!
   📋 Table     : `dev_team`.`testing`.Ajay_silver_flight_positions
   📊 Row count : 10676


In [0]:
# Final preview of Silver table
print("🔍 Silver Table — Sample Records:")
spark.table(SILVER_TABLE).select(
    "icao24", "callsign", "origin_country",
    "latitude", "longitude", "baro_altitude",
    "on_ground", "velocity", "api_response_time"
).show(10, truncate=25)

print("\n📊 Silver Table Stats:")
print(f"   Total aircraft    : {spark.table(SILVER_TABLE).count()}")
print(f"   Airborne          : {spark.table(SILVER_TABLE).filter('on_ground = false').count()}")
print(f"   On ground         : {spark.table(SILVER_TABLE).filter('on_ground = true').count()}")
print(f"   Unique countries  : {spark.table(SILVER_TABLE).select('origin_country').distinct().count()}")

print(f"\n🏁 Notebook 3 Complete!")

🔍 Silver Table — Sample Records:
+------+--------+--------------+--------+---------+-------------+---------+--------+-------------------+
|icao24|callsign|origin_country|latitude|longitude|baro_altitude|on_ground|velocity|  api_response_time|
+------+--------+--------------+--------+---------+-------------+---------+--------+-------------------+
|39de4f| TVF234P|        France| 37.6724|  -4.1238|      8633.46|    false|  232.15|2026-05-05 17:44:10|
|39de4e| TVF539X|        France| 48.6468|    4.416|       8724.9|    false|   207.4|2026-05-05 17:44:10|
|ab1644| UAL2041| United States| 35.9777| -93.4529|     11269.98|    false|  283.67|2026-05-05 17:44:10|
|a47559|  N387AP| United States| 34.0908| -84.4043|      1036.32|    false|   62.81|2026-05-05 17:44:10|
|39de4a| TVF87TP|        France|  33.724|  -7.7861|     11590.02|    false|  255.59|2026-05-05 17:44:10|
|39de4d| TVF67YK|        France| 41.7377|  -3.1548|      11582.4|    false|  244.46|2026-05-05 17:44:10|
|39de4c| TVF23QB|     